# Эксперименты с ABM «Отказ НПЗ и дефицит бензина» (refinery_shock_abm, NetLogo 7)

Общий ноутбук проекта. Этапы добавляются по очереди; каждый этап читает файлы предыдущего из `OUT_DIR`, поэтому ноутбук можно перезапускать с любого этапа.

| Этап | Что делаем | Главные файлы |
|---|---|---|
| **0. Подготовка** (этот блок) | проверки модели, seed, горизонт, время прогона, число повторов R, выбор плана скрининга, базовый прогон P0 | `seeds.json`, `factors.csv`, `pilot.csv`, `baseline_P0.csv`, `prep_summary.json` |
| 1. Скрининг | факториальный план по переключателям L2–L6 × 3 уровня `shock-depth` × R повторов (CRN), ANOVA, эффекты | `design_screening.csv`, `screening.csv` |
| 2. Оптимизация уровней | LHS по непрерывным параметрам оставшихся рычагов, отбор финалистов | `design_levels.csv`, `levels.csv` |
| 3. Устойчивость | финалисты × LHS неопределённости × `shock-depth` × повторы; среднее, худший случай, сожаление, чувствительность (PRCC) | `design_robust.csv`, `robust.csv` |
| 4. Постобработка | Парето-фронт, таблица «сценарий приоритетов → стратегия», карта весов | рисунки, таблицы |
| 5. Проверка | финалисты на новых seed | `validation.csv` |

**Как работает запуск.** Модуль `abm_runner.py` (создаётся ячейкой ниже) держит по одной JVM с загруженной моделью в каждом процессе. Перед каждым прогоном выставляются *все* элементы интерфейса (`clear-all` их не сбрасывает), затем `setup`, затем, если нужно, переопределение констант неопределённости, затем `repeat N [go]`. KPI читаются одним вызовом в контрольных точках (по умолчанию дни 30, 90, 180 и 365).

## 0.1. Окружение

Нужны NetLogo 7 (установленный, лицензия не нужна) и пакеты Python. Java отдельно ставить не нужно: используется JVM, встроенная в NetLogo (папка `runtime`).

In [ ]:
import json, hashlib, math, os, platform, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110,
                     "axes.prop_cycle": plt.cycler(color=C)})
print(sys.version.split()[0], platform.platform())

3.13.14 Windows-11-10.0.26200-SP0


## 0.2. Конфигурация

**Единственная ячейка, которую нужно править под свой компьютер:** пути к NetLogo и к модели. Остальное - решения по дизайну эксперимента (обоснование в комментариях; диапазоны - предложение, их нужно подтвердить источниками в отчёте).

In [7]:
# --- пути (Windows: r"C:\Program Files\NetLogo 7.0.3") ---
NETLOGO_HOME = r"C:\Program Files\NetLogo 7.0.4"
MODEL_PATH   = Path(r"refinery_shock_abm.nlogox")
OUT_DIR      = Path("results")
OUT_DIR.mkdir(exist_ok=True)

# --- время ---
HORIZON     = 365                      # дней (1 tick = 1 день), шок с 5-го дня
CHECKPOINTS = [30, 90, 180, HORIZON]   # дни, в которые снимаем накопленные KPI (фаза кризиса vs год)

# --- сценарная ось ---
# Поражённый НПЗ даёт ~1/3 рынка, поэтому потеря рынка ~ shock-depth / 3.
# Экспортный буфер normal-surplus = 6%: L2 закрывает потерю до ~6% рынка (shock-depth ~ 0.18).
# 0.2 -> ~6.7% (почти закрывается L2), 0.45 -> ~15%, 0.7 -> ~23% (без L3/L4 дефицит неизбежен)
SHOCK_LEVELS = [0.2, 0.45, 0.7]
SHOCK_MID    = 0.45

# --- рычаги ЛПР ---
LEVERS = {  # короткое имя -> переключатель интерфейса
    "L2": "L2-export-ban?", "L3": "L3-quality-relax?", "L4": "L4-import?",
    "L5": "L5-fillup-limit?", "L6": "L6-price-cap?",
}
LEVER_RANGES = {  # для этапа 2 (LHS по силе рычагов); 'off' эквивалентен краю диапазона
    "L3-share-naphtha":     (0.0, 0.5),    # 0 = L3 выключен
    "L4-import-limit":      (0.0, 0.3),    # 0 = L4 выключен (доля D0 в день)
    "L5-liters-per-fillup": (10, 70),      # >= 70 (бак 50 + канистра 20) = L5 не действует
    "L6-price-cap":         (1.0, 2.0),
}
LEVER_ON = None   # уровни 'on' для скрининга; None = значения слайдеров из файла модели
# L4-subsidy НЕ влияет на динамику (объём импорта от неё не зависит): K2 = L4-subsidy * объём импорта.
# Поэтому субсидию держим на значении из модели, а K2 при другой субсидии пересчитываем без прогонов.

# --- неопределённость (этап 3, LHS); номинал = константа модели ---
UNC_RANGES = {
    "panic-sensitivity": (0.5, 2.0),     # номинал 1.0
    "tau-down":          (7, 30),        # номинал 14 дней
    "w-own-experience":  (0.2, 0.8),     # номинал 0.5
    "price-elasticity":  (-0.4, -0.05),  # номинал -0.2; краткосрочная эластичность бензина
    "normal-surplus":    (0.02, 0.12),   # номинал 0.06
}
C3_RANGE = (0.05, 0.5)   # ущерб от низкосортного бензина, m.u./L; только постобработка: K3-cost = c3 * ref-price * K3-volume

# --- воспроизводимость ---
MASTER_SEED = 20261119   # генерирует все наборы seed; не менять после начала экспериментов
SEED_POOLS  = {"diag": 10, "pilot": 30, "screening": 50, "levels": 30, "robust": 30, "validation": 50}

# --- вычисления ---
N_WORKERS_MAX = max(1, (os.cpu_count() or 2) - 1)
JVM_ARGS      = ["-Xmx1g"]      # память на одну JVM (по процессу)

KPIS     = ["K0", "K1", "K2", "K3_volume", "K4", "K5_hours"]
KPI_NAME = {"K0": "физический дефицит, L", "K1": "спрос, вытесненный ценой, L",
            "K2": "субсидия импорта, m.u.", "K3_volume": "низкосортный бензин, L",
            "K4": "переплата потребителей, m.u.", "K5_hours": "очереди, машино-часы"}

## 0.3. Модуль запуска `abm_runner.py`

Ячейка записывает модуль на диск: рабочие процессы пула (spawn) импортируют функции из файла, а не из ноутбука. Это обязательно на Windows.

Ключевой момент - **переопределение констант**. `panic-sensitivity`, `tau-down` и др. задаются в `set-constants` внутри `setup`, поэтому выставить их до `setup` нельзя, а копию модели со слайдерами мы не делаем. Функция `override_command` меняет их после `setup` так, что результат совпадает с изменённой константой (обоснование в docstring; проверка - в разделе 2.2).

In [16]:
%%writefile abm_runner.py
"""
abm_runner.py - запуск модели refinery_shock_abm (NetLogo 7) из Python через pyNetLogo.

Один процесс = одна JVM = одна загруженная модель. Параллельность - через
ProcessPoolExecutor с контекстом "spawn" (JVM нельзя форкать; на Windows spawn и так по умолчанию).

Основные функции:
    make_link(cfg)                     - JVM + загрузка модели в текущем процессе
    run_one(link, job, cfg)            - один прогон, KPI в контрольных точках
    run_timeseries(link, job, cfg)     - один прогон с записью ряда по дням (диагностика)
    run_batch(jobs, cfg, n_workers)    - пакет прогонов, параллельно, с дозаписью и возобновлением
    override_command(unc, nominal)     - команда NetLogo, переопределяющая константы после setup
"""
from __future__ import annotations

import glob
import math
import os
import sys
import time
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import get_context
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Имена элементов интерфейса модели
# ---------------------------------------------------------------------------
SWITCHES = ["L2-export-ban?", "L3-quality-relax?", "L4-import?", "L5-fillup-limit?", "L6-price-cap?"]
SLIDERS = ["L3-share-naphtha", "L4-import-limit", "L4-subsidy", "L5-liters-per-fillup",
           "L6-price-cap", "shock-depth"]
# Константы из set-constants, которые меняются в LHS неопределённости (c3 - только постобработка)
UNCERTAIN = ["panic-sensitivity", "tau-down", "w-own-experience", "price-elasticity", "normal-surplus"]

# Что читаем в каждой контрольной точке (накопленные KPI + состояние)
KPI_REPORTERS = {
    "K0": "K0-total",
    "K1": "K1-total",
    "K2": "K2-total",
    "K3_volume": "K3-volume-total",
    "K4": "K4-total",
    "K5_hours": "K5-total-hours",
    "price": "price",
    "dry_share": "dry-share",
    "perceived": "mean-perceived-shortage",
    "failed": "failed-count",
}
# Что читаем один раз после setup (паспорт прогона)
SETUP_REPORTERS = {
    "seed_used": "seed-used",
    "policy_delay": "policy-delay",
    "capacity_loss": "market-capacity-loss",
    "D0": "D0",
}
# Ряд по дням для диагностики
TS_REPORTERS = [
    "ticks", "price", "dry-share", "production-today", "export-today", "import-today",
    "sales-today", "consumption-today", "K0-today", "K1-today", "desired-today",
    "failed-count", "mean-perceived-shortage",
    "total-refinery-stock", "total-pipeline-stock", "total-station-stock", "total-consumer-stock",
    "K0-total", "K1-total", "K2-total", "K3-volume-total", "K4-total", "K5-total-hours",
]
CONSTANT_REPORTERS = [
    "shock-start", "policy-delay", "L3-ramp-days", "L4-import-delay",
    "n-refineries", "n-stations", "n-consumers", "normal-surplus", "naphtha-ratio",
    "refinery-storage-days", "transport-delay", "liters-per-car-day", "tank-liters",
    "canister-liters", "reorder-threshold", "price-elasticity", "panic-sensitivity",
    "w-own-experience", "tau-up", "tau-down", "base-search-radius", "radius-growth",
    "max-radius", "price-adj-speed", "ref-price", "price-coverage-coef",
    "service-reserve", "normal-visits", "station-service-capacity", "D0", "target-coverage",
]


def nl_value(v) -> str:
    """Python-значение -> литерал NetLogo."""
    if isinstance(v, (bool, np.bool_)):
        return "true" if v else "false"
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    return repr(float(v))


def nl_list(reporters) -> str:
    return "(list " + " ".join(reporters) + ")"


# ---------------------------------------------------------------------------
# Подключение
# ---------------------------------------------------------------------------
def find_jvm(netlogo_home: str | None) -> str | None:
    """JVM, встроенная в NetLogo 7 (папка runtime). Совместимая версия Java гарантирована."""
    if not netlogo_home:
        return None
    names = ["jvm.dll", "libjvm.so", "libjvm.dylib"]
    for name in names:
        hits = glob.glob(os.path.join(netlogo_home, "**", name), recursive=True)
        hits = [h for h in hits if "server" in h.replace("\\", "/")] or hits
        if hits:
            return hits[0]
    return None


def make_link(cfg: dict):
    """Стартует JVM в текущем процессе и загружает модель. Возвращает pynetlogo.NetLogoLink."""
    import pynetlogo
    jvm = cfg.get("jvm_path") or find_jvm(cfg.get("netlogo_home"))
    link = pynetlogo.NetLogoLink(
        gui=False,
        netlogo_home=cfg.get("netlogo_home"),
        jvm_path=jvm,
        jvmargs=cfg.get("jvmargs", []),
    )
    link.load_model(str(cfg["model_path"]))
    return link


# ---------------------------------------------------------------------------
# Переопределение констант после setup
# ---------------------------------------------------------------------------
def override_command(unc: dict | None, nominal: dict) -> str:
    """
    Константы задаются в set-constants внутри setup (после clear-all), поэтому их нельзя
    выставить до setup. Переопределяем после setup, так, чтобы результат совпадал с тем,
    как если бы константа была другой с самого начала:
      * tau-down, w-own-experience, price-elasticity в setup не используются -> просто set;
      * panic-sensitivity: panic-i = max(0, N(ps, 0.5 ps)) = ps * max(0, N(1, 0.5)),
        т. е. линеен по ps -> panic-i := panic-i * ps_new / ps_old (те же случайные числа);
      * normal-surplus: capacity = (1 + ns) * sum(base-sales) -> масштабируем capacity и
        пересчитываем naphtha-capacity, storage-cap и начальный stock НПЗ (stock = 0.5 * storage-cap).
    Всё внутри with-local-randomness: основной генератор не сдвигается, траектория
    случайных чисел та же, что без переопределения (общие случайные числа сохраняются).
    Значения, равные номиналу, не трогаем: тогда прогон побитово совпадает с прогоном без переопределения.
    """
    if not unc:
        return ""
    parts = []
    for k in ("tau-down", "w-own-experience", "price-elasticity"):
        if k in unc and not math.isclose(float(unc[k]), float(nominal[k]), rel_tol=0, abs_tol=1e-15):
            parts.append(f"set {k} {nl_value(unc[k])}")
    if "panic-sensitivity" in unc and unc["panic-sensitivity"] != nominal["panic-sensitivity"]:
        ps = nl_value(unc["panic-sensitivity"])
        parts.append(
            f"if panic-sensitivity > 0 [ ask consumers [ set panic-i panic-i * {ps} / panic-sensitivity ] ] "
            f"set panic-sensitivity {ps}"
        )
    if "normal-surplus" in unc and unc["normal-surplus"] != nominal["normal-surplus"]:
        ns = nl_value(unc["normal-surplus"])
        parts.append(
            f"ask refineries [ set capacity capacity * (1 + {ns}) / (1 + normal-surplus) "
            f"set naphtha-capacity capacity * naphtha-ratio "
            f"set storage-cap refinery-storage-days * capacity "
            f"set stock 0.5 * storage-cap ] "
            f"set normal-surplus {ns}"
        )
    unknown = set(unc) - set(UNCERTAIN)
    if unknown:
        raise ValueError(f"Неизвестные параметры неопределённости: {unknown}")
    if not parts:
        return ""
    return "with-local-randomness [ " + " ".join(parts) + " ]"


# ---------------------------------------------------------------------------
# Один прогон
# ---------------------------------------------------------------------------
def _prepare(link, job: dict, cfg: dict) -> dict:
    """Выставляет ВСЕ элементы интерфейса (clear-all их не сбрасывает), seed, setup, оверрайды."""
    params = {**cfg["interface_defaults"], **job.get("params", {})}
    cmd = [f"set {name} {nl_value(params[name])}" for name in SWITCHES + SLIDERS]
    seed = int(job["seed"])
    if seed == 0:
        raise ValueError("seed = 0 в модели означает случайное зерно - не используем")
    cmd.append(f"set seed {seed}")
    link.command(" ".join(cmd))
    link.command("setup")
    ov = override_command(job.get("uncertain"), cfg["nominal_uncertain"])
    if ov:
        link.command(ov)
    vals = link.report(nl_list(SETUP_REPORTERS.values()))
    return dict(zip(SETUP_REPORTERS.keys(), np.atleast_1d(vals).tolist()))


def run_one(link, job: dict, cfg: dict) -> dict:
    """Прогон до горизонта; KPI читаются в контрольных точках cfg['checkpoints'] (последняя = горизонт)."""
    t0 = time.perf_counter()
    out = {k: v for k, v in job.items() if k not in ("params", "uncertain")}
    out.update({f"p:{k}": v for k, v in job.get("params", {}).items()})
    out.update({f"u:{k}": v for k, v in (job.get("uncertain") or {}).items()})
    out.update(_prepare(link, job, cfg))
    horizon = int(cfg["horizon"])
    cps = sorted(set(int(c) for c in cfg["checkpoints"] if c < horizon)) + [horizon]
    done = 0
    rep = nl_list(KPI_REPORTERS.values())
    for cp in cps:
        link.command(f"repeat {cp - done} [ go ]")
        done = cp
        vals = np.atleast_1d(link.report(rep)).tolist()
        suffix = "" if cp == horizon else f"@{cp}"
        for k, v in zip(KPI_REPORTERS.keys(), vals):
            out[k + suffix] = float(v)
    out["sec"] = time.perf_counter() - t0
    out["pid"] = os.getpid()
    return out


def run_timeseries(link, job: dict, cfg: dict, ticks: int | None = None) -> pd.DataFrame:
    """Прогон с чтением состояния каждый день (строка 0 - сразу после setup). Только для диагностики."""
    _prepare(link, job, cfg)
    ticks = int(ticks or cfg["horizon"])
    rep = nl_list(TS_REPORTERS)
    rows = [np.atleast_1d(link.report(rep)).tolist()]
    for _ in range(ticks):
        link.command("go")
        rows.append(np.atleast_1d(link.report(rep)).tolist())
    return pd.DataFrame(rows, columns=TS_REPORTERS)


# ---------------------------------------------------------------------------
# Пакетный запуск
# ---------------------------------------------------------------------------
_LINK = None
_CFG = None


def _init_worker(cfg: dict):
    global _LINK, _CFG
    _CFG = cfg
    _LINK = make_link(cfg)


def _worker_task(job: dict) -> dict:
    try:
        return run_one(_LINK, job, _CFG)
    except Exception:
        return {**{k: v for k, v in job.items() if k not in ("params", "uncertain")},
                "error": traceback.format_exc()}


def run_batch(jobs: list[dict], cfg: dict, n_workers: int = 1, out_csv: str | Path | None = None,
              resume: bool = True, link=None, flush_every: int = 50, progress: bool = True) -> pd.DataFrame:
    """
    Пакет прогонов. Каждое задание - dict с ключами run_id (уникальный), seed, params, uncertain
    (+ любые метки: stage, design_id, rep ...).
    n_workers = 1 -> прогон в текущем процессе (нужен link); > 1 -> пул процессов, по JVM в каждом.
    out_csv: результаты дописываются каждые flush_every прогонов; при resume=True уже посчитанные
    run_id пропускаются (можно прервать и продолжить).
    """
    out_csv = Path(out_csv) if out_csv else None
    done_df = None
    if out_csv is not None and resume and out_csv.exists():
        done_df = pd.read_csv(out_csv)
        if "error" in done_df:
            done_df = done_df[done_df["error"].isna()]
        done_ids = set(done_df["run_id"])
        jobs = [j for j in jobs if j["run_id"] not in done_ids]
        print(f"[run_batch] уже посчитано {len(done_ids)}, осталось {len(jobs)}")
    ids = [j["run_id"] for j in jobs]
    if len(ids) != len(set(ids)):
        raise ValueError("run_id в заданиях не уникальны")

    results, buffer = [], []

    def flush():
        nonlocal buffer
        if out_csv is not None and buffer:
            df = pd.DataFrame(buffer)
            df.to_csv(out_csv, mode="a", header=not out_csv.exists(), index=False)
        buffer = []

    try:
        from tqdm.auto import tqdm
    except ImportError:  # без tqdm - без прогресс-бара
        def tqdm(x, **kw):
            return x

    t_start = time.perf_counter()
    if n_workers <= 1:
        if link is None:
            raise ValueError("Для n_workers = 1 передайте link (make_link(cfg))")
        it = tqdm(jobs, total=len(jobs), disable=not progress)
        for job in it:
            try:
                r = run_one(link, job, cfg)
            except Exception:
                r = {**{k: v for k, v in job.items() if k not in ("params", "uncertain")},
                     "error": traceback.format_exc()}
            results.append(r)
            buffer.append(r)
            if len(buffer) >= flush_every:
                flush()
    else:
        ctx = get_context("spawn")
        with ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx,
                                 initializer=_init_worker, initargs=(cfg,)) as ex:
            futs = [ex.submit(_worker_task, j) for j in jobs]
            for f in tqdm(as_completed(futs), total=len(futs), disable=not progress):
                r = f.result()
                results.append(r)
                buffer.append(r)
                if len(buffer) >= flush_every:
                    flush()
    flush()
    wall = time.perf_counter() - t_start
    df = pd.DataFrame(results)
    n_err = int(df["error"].notna().sum()) if "error" in df else 0
    if len(jobs):
        print(f"[run_batch] {len(jobs)} прогонов за {wall / 60:.1f} мин "
              f"({len(jobs) / max(wall, 1e-9) * 3600:.0f} прогонов/ч), ошибок: {n_err}")
    if n_err:
        print(df.loc[df["error"].notna(), "error"].iloc[0])
    if done_df is not None and len(done_df):
        df = pd.concat([done_df, df], ignore_index=True)
    df.attrs["wall_sec"] = wall
    return df

Overwriting abm_runner.py


## 0.4. Подключение к NetLogo и паспорт модели

Читаем из модели значения слайдеров (уровни рычагов по умолчанию) и все константы после `setup`. SHA-256 файла модели фиксирует версию: если файл изменится, результаты разных этапов нельзя смешивать.

In [ ]:
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
import importlib, abm_runner as ar
importlib.reload(ar)

MODEL_SHA256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
BASE_CFG = {"netlogo_home": NETLOGO_HOME, "model_path": str(MODEL_PATH.resolve()),
            "jvmargs": JVM_ARGS, "jvm_path": ar.find_jvm(NETLOGO_HOME),
            "horizon": HORIZON, "checkpoints": CHECKPOINTS}
print("JVM:", BASE_CFG["jvm_path"])

link = ar.make_link(BASE_CFG)          # JVM в процессе ноутбука (для диагностики и n_workers = 1)
NETLOGO_VERSION = link.report("netlogo-version")
print("NetLogo", NETLOGO_VERSION, "| модель sha256:", MODEL_SHA256[:16], "...")

slider_vals = dict(zip(ar.SLIDERS, np.atleast_1d(link.report(ar.nl_list(ar.SLIDERS))).tolist()))
INTERFACE_DEFAULTS = {sw: False for sw in ar.SWITCHES} | slider_vals   # P0: все переключатели off
INTERFACE_DEFAULTS["shock-depth"] = SHOCK_MID
if LEVER_ON is None:
    LEVER_ON = {k: slider_vals[k] for k in LEVER_RANGES}
print("Слайдеры в файле модели:", slider_vals)
print("Уровни 'on' для скрининга:", LEVER_ON)

# константы после setup (seed любой - от него зависит только policy-delay)
link.command(" ".join(f"set {k} {ar.nl_value(v)}" for k, v in INTERFACE_DEFAULTS.items()) + " set seed 1 setup")
CONSTANTS = dict(zip(ar.CONSTANT_REPORTERS,
                     np.atleast_1d(link.report(ar.nl_list(ar.CONSTANT_REPORTERS))).tolist()))
NOMINAL_UNC = {k: CONSTANTS[k] for k in ar.UNCERTAIN}
print("Номинал неопределённых параметров:", NOMINAL_UNC)

CFG = BASE_CFG | {"interface_defaults": INTERFACE_DEFAULTS, "nominal_uncertain": NOMINAL_UNC}
pd.Series(CONSTANTS, name="значение").to_frame()

JVM: C:\Program Files\NetLogo 7.0.4\runtime\bin\server\jvm.dll


RuntimeError: Loaded unexpected abm_runner module: c:\Users\arsmi\Downloads\netlogo-project\Experiments\abm_runner.py. Re-run the %%writefile abm_runner.py cell and restart the kernel if needed.

In [15]:
from pathlib import Path
p = Path(r"c:\Users\arsmi\Downloads\netlogo-project\Experiments\abm_runner.py")
txt = p.read_text(encoding="utf-8")
print(len(txt), "bytes; mtime:", p.stat().st_mtime)
print("make_link в тексте:", "def make_link" in txt)
print(txt[:400])

3272 bytes; mtime: 1795020203.9267633
make_link в тексте: False
"""
abm_runner.py - запуск модели refinery_shock_abm (NetLogo 7) из Python через pyNetLogo.

Один процесс = одна JVM = одна загруженная модель. Параллельность - через
ProcessPoolExecutor с контекстом "spawn" (JVM нельзя форкать; на Windows spawn и так по умолчанию).

Основные функции:
    make_link(cfg)                     - JVM + загрузка модели в текущем процессе
    run_one(link, job, cfg)     


**Если подключение не сработало.**
* `JVMNotFoundException` / не найден `jvm.dll` - укажите `BASE_CFG["jvm_path"]` вручную (обычно `...\NetLogo 7.x\runtime\bin\server\jvm.dll`).
* `JVM is already started` после правки конфигурации - перезапустите ядро (одна JVM на процесс, перезапустить её нельзя).
* `ClassNotFoundException`, `NoSuchMethodError` или ошибка при `load_model` - вероятна несовместимость pyNetLogo (последний релиз собран под NetLogo 6.x) с NetLogo 7. Пришлите текст ошибки: запасной вариант - запуск через headless BehaviorSpace (`NetLogo_Console --headless`), его можно встроить в `abm_runner` без изменения остальных ячеек.

### Таблица факторов

Фиксируется до экспериментов и идёт в отчёт (раздел Experimental design). Роли: **рычаг** - выбирает регулятор; **сценарий** - внешняя ось, по которой результаты показываются отдельно; **неопределённость** - параметры, к которым стратегия должна быть устойчива; **постобработка** - не влияют на динамику, пересчитываются по готовым прогонам.

In [ ]:
rows = [{"фактор": sw, "роль": "рычаг (on/off)", "номинал": "off", "диапазон / уровни": "off / on",
         "этапы": "1-5"} for sw in ar.SWITCHES]
rows += [{"фактор": k, "роль": "рычаг (сила)", "номинал": LEVER_ON[k], "диапазон / уровни": f"{lo} ... {hi}",
          "этапы": "2-5 (в 1 - уровень 'on')"} for k, (lo, hi) in LEVER_RANGES.items()]
rows += [{"фактор": "shock-depth", "роль": "сценарий", "номинал": SHOCK_MID,
          "диапазон / уровни": " / ".join(map(str, SHOCK_LEVELS)), "этапы": "все"}]
rows += [{"фактор": k, "роль": "неопределённость", "номинал": NOMINAL_UNC[k],
          "диапазон / уровни": f"{lo} ... {hi} (LHS)", "этапы": "3, 5"} for k, (lo, hi) in UNC_RANGES.items()]
rows += [{"фактор": "c3", "роль": "неопределённость (постобработка)", "номинал": "-",
          "диапазон / уровни": f"{C3_RANGE[0]} ... {C3_RANGE[1]}", "этапы": "3-5, без прогонов"},
         {"фактор": "L4-subsidy", "роль": "рычаг (постобработка)", "номинал": slider_vals["L4-subsidy"],
          "диапазон / уровни": "K2 = L4-subsidy x объём импорта", "этапы": "4"},
         {"фактор": "policy-delay", "роль": "внутренний шум (из seed)", "номинал": "7-10 дней",
          "диапазон / уровни": "random 4 в setup", "этапы": "все"}]
FACTORS = pd.DataFrame(rows)
FACTORS.to_csv(OUT_DIR / "factors.csv", index=False, encoding="utf-8-sig")
FACTORS

## 1. Наборы seed

Все seed берутся из одного генератора `MASTER_SEED` и делятся на **непересекающиеся** пулы: проверка на этапе 5 идёт на seed, которых модель «не видела». Внутри этапа все комбинации рычагов используют одни и те же seed (общие случайные числа, CRN): повтор r = seed r. Слайдер `seed` в модели 0-1000, `seed = 0` означает случайное зерно, поэтому берём 1-1000.

CRN в ABM частичные: `setup` (город, машины, `policy-delay`) у всех стратегий одинаков, но после шока разные стратегии тратят разное количество случайных чисел, и потоки расходятся. Насколько CRN всё-таки уменьшает дисперсию разниц, видно в пилоте (раздел 4).

In [ ]:
_rng = np.random.default_rng(MASTER_SEED)
_perm = _rng.permutation(np.arange(1, 1001)).tolist()
SEEDS, i = {}, 0
for name, n in SEED_POOLS.items():
    SEEDS[name] = _perm[i:i + n]
    i += n
assert len(set(sum(SEEDS.values(), []))) == i, "пулы пересекаются"
(OUT_DIR / "seeds.json").write_text(json.dumps({"master_seed": MASTER_SEED, "pools": SEEDS}, indent=1))
{k: v[:5] for k, v in SEEDS.items()}

In [ ]:
# Политики для проверок и пилота (P1, P2 - пакеты из постановки)
def policy(*levers, **strength):
    p = {LEVERS[l]: True for l in levers}
    p.update(LEVER_ON)
    p.update(strength)
    return p

POLICIES = {"P0": policy(), "P1": policy("L3", "L5", "L6"), "P2": policy("L2", "L4"),
            "ALL": policy("L2", "L3", "L4", "L5", "L6")}

def job(seed, pol="P0", shock=SHOCK_MID, unc=None, stage="diag", design_id=None, tag=None, **labels):
    # pol: имя из POLICIES или dict параметров интерфейса (тогда нужен design_id)
    if isinstance(pol, str):
        params, design_id = POLICIES[pol] | {"shock-depth": shock}, pol
    else:
        params = pol | {"shock-depth": shock}
    run_id = f"{stage}|{design_id}|sd={shock}|seed={seed}" + (f"|{tag}" if tag else "")
    return {"run_id": run_id, "stage": stage, "design_id": design_id, "shock": shock, "seed": int(seed),
            "params": params, "uncertain": unc, **labels}

CHECKS = {}
def check(name, ok, detail=""):
    CHECKS[name] = {"ok": bool(ok), "detail": str(detail)}
    print(("PASS " if ok else "FAIL ") + name + (f" | {detail}" if detail else ""))

## 2. Проверки модели

Модель замораживается после этих проверок. Проверки C1 и C5 повторяют приёмочные проверки модели, но уже в связке Python-NetLogo, на которой будут идти все эксперименты.

### 2.1. Воспроизводимость (C5) и невмешательство чтения

Один seed дважды подряд должен дать побитово одинаковые KPI. Ежедневное чтение состояния (`run_timeseries`) не должно менять траекторию: все репортёры модели обёрнуты в `with-local-randomness`.

In [ ]:
s = SEEDS["diag"][0]
j = job(s, "ALL")
a, b = ar.run_one(link, j, CFG), ar.run_one(link, j, CFG)
ts = ar.run_timeseries(link, j, CFG)
same = all(a[k] == b[k] for k in KPIS)
check("C5: тот же seed -> те же KPI", same, {k: a[k] for k in KPIS})
same_ts = all(np.isclose(ts.iloc[-1][r], a[k], rtol=0, atol=0)
              for k, r in [("K0", "K0-total"), ("K4", "K4-total"), ("K5_hours", "K5-total-hours")])
check("чтение каждый день не меняет траекторию", same_ts)
print(f"один прогон {HORIZON} дней: {a['sec']:.1f} с")

### 2.2. Переопределение констант неопределённости

(a) Номинальные значения через `override_command` дают тот же прогон, что и без переопределения (побитово).
(b) `panic-sensitivity` масштабирует `panic-i`; `normal-surplus` масштабирует мощность НПЗ, а случайный поток не сдвигается (`policy-delay` и город те же).
(c) Направление эффектов разумно: сильнее паника -> больше дефицит; больше `normal-surplus` -> меньше дефицит при P0.

In [ ]:
s = SEEDS["diag"][1]
base = ar.run_one(link, job(s, "P0"), CFG)
nom = ar.run_one(link, job(s, "P0", unc=dict(NOMINAL_UNC), tag="nominal"), CFG)
check("(a) оверрайд номиналом = без оверрайда", all(base[k] == nom[k] for k in KPIS))

def probe(unc):
    j = job(s, "P0", unc=unc, tag="probe")
    ar._prepare(link, j, CFG)
    r = link.report("(list mean [panic-i] of consumers  sum [capacity] of refineries  policy-delay  "
                    "[xcor] of min-one-of consumers [who])")
    return np.atleast_1d(r).tolist()

p0 = probe(None)
p1 = probe({"panic-sensitivity": 2.0, "normal-surplus": 0.12})
check("(b) panic-i x2", math.isclose(p1[0] / p0[0], 2.0 / NOMINAL_UNC["panic-sensitivity"], rel_tol=1e-9),
      f"{p0[0]:.4f} -> {p1[0]:.4f}")
check("(b) capacity x (1.12/1.06)", math.isclose(p1[1] / p0[1], 1.12 / (1 + NOMINAL_UNC["normal-surplus"]), rel_tol=1e-9))
check("(b) случайный поток не сдвинут (policy-delay, координаты)", p0[2] == p1[2] and p0[3] == p1[3])

lo = ar.run_one(link, job(s, "P0", unc={"panic-sensitivity": 0.5}, tag="lo"), CFG)
hi = ar.run_one(link, job(s, "P0", unc={"panic-sensitivity": 2.0}, tag="hi"), CFG)
check("(c) паника 2.0 даёт больший K0, чем 0.5", hi["K0"] > lo["K0"], f"{lo['K0']:.0f} vs {hi['K0']:.0f}")
# normal-surplus при P0 почти не меняет внутреннее предложение (излишек и так уходит на экспорт),
# он работает через L2: запрет экспорта освобождает больше. Поэтому проверяем на политике «только L2».
l2 = policy("L2")
ns_lo = ar.run_one(link, job(s, l2, design_id="L2", unc={"normal-surplus": 0.02}, tag="ns_lo"), CFG)
ns_hi = ar.run_one(link, job(s, l2, design_id="L2", unc={"normal-surplus": 0.12}, tag="ns_hi"), CFG)
check("(c) при L2: normal-surplus 0.12 даёт меньший K0, чем 0.02", ns_hi["K0"] < ns_lo["K0"],
      f"{ns_lo['K0']:.0f} vs {ns_hi['K0']:.0f}")

### 2.3. Норма без шока (C1)

`shock-depth = 0`, все меры выключены: цена 1 ± 0.03 после 20-го дня, дефицита нет, доля пустых АЗС < 5%.

In [ ]:
ts0 = ar.run_timeseries(link, job(SEEDS["diag"][2], "P0", shock=0.0), CFG)
after = ts0[ts0.ticks >= 20]
check("C1: цена 1 +- 0.03 после 20-го дня", after.price.between(0.97, 1.03).all(),
      f"min {after.price.min():.3f}, max {after.price.max():.3f}")
check("C1: K0 ~ 0", ts0["K0-total"].iloc[-1] < 1e-6 * CONSTANTS["D0"] * HORIZON, f"K0 = {ts0['K0-total'].iloc[-1]:.1f} L")
check("C1: dry-share < 0.05", ts0["dry-share"].max() < 0.05, f"max {ts0['dry-share'].max():.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(ts0.ticks, ts0.price); ax[0].set_title("Цена (индекс), без шока"); ax[0].set_xlabel("день")
ax[1].plot(ts0.ticks, ts0["dry-share"]); ax[1].set_title("Доля пустых АЗС, без шока"); ax[1].set_xlabel("день")
plt.tight_layout(); plt.show()

### 2.4. Баланс массы

Для прогона с шоком и всеми мерами: изменение суммарных запасов (НПЗ + в пути + АЗС + машины) за день равно производство − экспорт + импорт − фактическое потребление (потребление − K0 за день). Остаток должен быть на уровне ошибки округления.

In [ ]:
tsb = ar.run_timeseries(link, job(SEEDS["diag"][3], "ALL", shock=SHOCK_LEVELS[-1]), CFG)
stock = tsb[["total-refinery-stock", "total-pipeline-stock", "total-station-stock", "total-consumer-stock"]].sum(axis=1)
flow = (tsb["production-today"] - tsb["export-today"] + tsb["import-today"]
        - (tsb["consumption-today"] - tsb["K0-today"]))
resid = (stock.diff() - flow).iloc[1:]
rel = resid.abs().max() / CONSTANTS["D0"]
check("баланс массы", rel < 1e-6, f"max |остаток| = {resid.abs().max():.2e} L ({rel:.1e} D0)")

### 2.5. Горизонт: что накапливается за 365 дней

Шок постоянный, поэтому за год накопленные KPI в основном отражают **новый устойчивый режим**, а не острую фазу паники. Таблица показывает, какая доля годового значения набирается к 30/90/180-му дню. Если острая фаза важна для выводов, на этапах анализа используем и KPI на день 90 (они уже пишутся в контрольных точках, столбцы `K0@90` и т. д.).

In [ ]:
diag = {}
for shock in SHOCK_LEVELS:
    for pol in ["P0", "ALL"]:
        diag[(shock, pol)] = ar.run_timeseries(link, job(SEEDS["diag"][4], pol, shock=shock), CFG)

fig, axes = plt.subplots(3, 3, figsize=(13, 8), sharex=True)
for col, shock in enumerate(SHOCK_LEVELS):
    for i, pol in enumerate(["P0", "ALL"]):
        d = diag[(shock, pol)]
        axes[0, col].plot(d.ticks, d.price, color=C[i], label=pol)
        axes[1, col].plot(d.ticks, d["dry-share"], color=C[i], label=pol)
        axes[2, col].plot(d.ticks, d["K0-today"] / CONSTANTS["D0"], color=C[i], label=pol)
    axes[0, col].set_title(f"shock-depth = {shock}")
axes[0, 0].set_ylabel("цена (индекс)"); axes[1, 0].set_ylabel("доля пустых АЗС")
axes[2, 0].set_ylabel("дефицит за день / D0")
for a in axes[2]: a.set_xlabel("день")
axes[0, 0].legend(frameon=False)
plt.tight_layout(); plt.show()

rows = []
for (shock, pol), d in diag.items():
    r = {"shock": shock, "policy": pol}
    for k, col in [("K0", "K0-total"), ("K4", "K4-total"), ("K5", "K5-total-hours")]:
        tot = d[col].iloc[-1]
        for cp in CHECKPOINTS[:-1]:
            r[f"{k}: доля к дню {cp}"] = d[col].iloc[cp] / tot if tot > 0 else np.nan
    rows.append(r)
pd.DataFrame(rows).round(2)

## 3. Время прогона и параллельность

Меряем время одного прогона в процессе ноутбука и пропускную способность пула при разном числе процессов (в каждом своя JVM, старт JVM и загрузка модели входят в замер). Дальше бюджет всех этапов считается по лучшей пропускной способности.

In [ ]:
t_single = [ar.run_one(link, job(s, "ALL", tag="t"), CFG)["sec"] for s in SEEDS["diag"][5:8]]
print(f"один прогон: {np.mean(t_single):.2f} с (по {len(t_single)} прогонам)")

worker_grid = sorted({1, 2, max(1, N_WORKERS_MAX // 2), N_WORKERS_MAX} - {1})
timing = [{"workers": 1, "runs_per_hour": 3600 / np.mean(t_single)}]
for n in worker_grid:
    jobs = [job(SEEDS["pilot"][k % len(SEEDS["pilot"])], "ALL", tag=f"w{n}k{k}") for k in range(3 * n)]
    df = ar.run_batch(jobs, CFG, n_workers=n, progress=False)
    timing.append({"workers": n, "runs_per_hour": len(jobs) / df.attrs["wall_sec"] * 3600})
TIMING = pd.DataFrame(timing)
TIMING.to_csv(OUT_DIR / "timing.csv", index=False)
best = TIMING.loc[TIMING.runs_per_hour.idxmax()]
N_WORKERS, RUNS_PER_HOUR = int(best.workers), float(best.runs_per_hour)
print(f"выбрано: {N_WORKERS} процессов, ~{RUNS_PER_HOUR:.0f} прогонов/ч (с учётом старта JVM - оценка снизу)")
TIMING.round(0)

## 4. Пилот: сколько нужно повторов R

P0, P1, P2 и ALL при `shock-depth = 0.45`, по 30 общих seed. Правило выбора R:

* для KPI ущерба (K0, K1, K4, K5) полуширина 95%-го ДИ **парной** разницы «политика − P0» должна быть не больше δ = 5% от среднего P0 (ищем различия, существенные относительно масштаба кризиса);
* для K2 и K3 (у P0 они равны 0) полуширина ДИ среднего не больше 5% самого среднего.

R = максимум по политикам и KPI, округление вверх до 5, в пределах [10, 50]. Корреляция KPI между политиками по seed показывает, насколько работают общие случайные числа (чем ближе к 1, тем сильнее CRN сжимают дисперсию разниц).

In [ ]:
pilot_jobs = [job(s, pol, stage="pilot") for pol in POLICIES for s in SEEDS["pilot"]]
PILOT = ar.run_batch(pilot_jobs, CFG, n_workers=N_WORKERS, link=link, out_csv=OUT_DIR / "pilot.csv")
PILOT = PILOT.sort_values(["design_id", "seed"]).reset_index(drop=True)

# пропускная способность в установившемся режиме (без старта JVM, с учётом конкуренции процессов за CPU)
RUNS_PER_HOUR = N_WORKERS * 3600 / PILOT["sec"].mean()
print(f"среднее время прогона в пуле {PILOT['sec'].mean():.1f} с -> ~{RUNS_PER_HOUR:.0f} прогонов/ч на {N_WORKERS} процессах")
with np.errstate(invalid="ignore", divide="ignore"):
    cv = (PILOT.groupby("design_id")[KPIS].std() / PILOT.groupby("design_id")[KPIS].mean()).round(3)
print("Коэффициент вариации по seed:"); display(cv)

In [ ]:
DELTA, ALPHA, R_MIN, R_MAX = 0.05, 0.05, 10, 50

def r_needed(sd, target, r_lo=3, r_hi=500):
    # минимальное R: t_{1-a/2, R-1} * sd / sqrt(R) <= target
    if not np.isfinite(sd) or target <= 0:
        return np.nan
    for R in range(r_lo, r_hi + 1):
        if stats.t.ppf(1 - ALPHA / 2, R - 1) * sd / np.sqrt(R) <= target:
            return R
    return r_hi

wide = PILOT.pivot(index="seed", columns="design_id", values=KPIS)
req, corr = [], []
for pol in ["P1", "P2", "ALL"]:
    for k in KPIS:
        if k in ("K2", "K3_volume"):
            x = wide[(k, pol)]
            R = r_needed(x.std(), DELTA * abs(x.mean())) if x.mean() > 0 else np.nan
            kind = "среднее"
        else:
            d = wide[(k, pol)] - wide[(k, "P0")]
            R = r_needed(d.std(), DELTA * wide[(k, "P0")].mean()) if wide[(k, "P0")].mean() > 0 else np.nan
            kind = "разница с P0"
            with np.errstate(invalid="ignore", divide="ignore"):
                corr.append({"policy": pol, "KPI": k, "corr с P0 по seed": wide[(k, pol)].corr(wide[(k, "P0")])})
        req.append({"policy": pol, "KPI": k, "критерий": kind, "R": R})
REQ = pd.DataFrame(req).pivot(index="KPI", columns="policy", values="R")
R_SCREEN = int(np.clip(5 * math.ceil(np.nanmax(REQ.values) / 5), R_MIN, R_MAX))
print("Требуемое R по политикам и KPI:"); display(REQ)
print("CRN: корреляция с P0 по seed"); display(pd.DataFrame(corr).pivot(index="KPI", columns="policy").round(2))
print(f"R_SCREEN = {R_SCREEN}" + ("  (упёрлись в верхнюю границу - смотрите, какой KPI шумит)" if R_SCREEN == R_MAX else ""))

In [ ]:
# Полуширина ДИ разницы ALL - P0 в % от среднего P0 как функция R (по дисперсии пилота)
Rs = np.arange(5, 51)
fig, ax = plt.subplots(figsize=(7, 3.5))
for i, k in enumerate(["K0", "K1", "K4", "K5_hours"]):
    sd = (wide[(k, "ALL")] - wide[(k, "P0")]).std()
    m0 = wide[(k, "P0")].mean()
    if m0 > 0:
        ax.plot(Rs, 100 * stats.t.ppf(1 - ALPHA / 2, Rs - 1) * sd / np.sqrt(Rs) / m0, color=C[i], label=k)
ax.axhline(100 * DELTA, color="#52514e", lw=1, ls="--"); ax.axvline(R_SCREEN, color="#52514e", lw=1, ls=":")
ax.set_xlabel("число повторов R"); ax.set_ylabel("полуширина 95% ДИ, % от P0")
ax.set_title("Точность разницы ALL - P0 при shock-depth = 0.45"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 5. Бюджет и выбор плана скрининга

Полный факториал 2⁵ = 32 комбинации против полуреплики 2⁵⁻¹ разрешения V (16 комбинаций, генератор E = ABCD; главные эффекты и все парные взаимодействия не смешаны между собой, смешаны только с взаимодействиями 3-го и 4-го порядка). Правило: **если полный факториал укладывается в бюджет времени скрининга, берём его** - он оценивает и тройные взаимодействия, а их в нелинейной модели с порогами исключать заранее нельзя. Числа для этапов 2-5 - ориентировочные, их уточним на своих этапах.

In [ ]:
SCREEN_BUDGET_H = 4.0   # сколько часов готовы отдать скринингу

import itertools
full = np.array(list(itertools.product([-1, 1], repeat=5)))
half = full[full[:, 4] == full[:, 0] * full[:, 1] * full[:, 2] * full[:, 3]]   # E = ABCD
# проверка разрешения V: главные эффекты и 10 парных взаимодействий попарно ортогональны
X = np.column_stack([half] + [half[:, i] * half[:, j] for i, j in itertools.combinations(range(5), 2)])
check("2^(5-1): главные эффекты и парные взаимодействия не смешаны", np.allclose(X.T @ X, len(half) * np.eye(X.shape[1])))

nS = len(SHOCK_LEVELS)
plan = pd.DataFrame([
    {"этап": "1. скрининг, полный 2^5", "прогонов": 32 * nS * R_SCREEN},
    {"этап": "1. скрининг, 2^(5-1) V", "прогонов": 16 * nS * R_SCREEN},
    {"этап": "2. уровни: 4 семейства x 60 LHS x 10 seed", "прогонов": 4 * 60 * nS * 10},
    {"этап": "3. устойчивость: 9 стратегий x 100 LHS x 3 seed", "прогонов": 9 * 100 * nS * 3},
    {"этап": "5. проверка: 9 стратегий x 30 новых seed", "прогонов": 9 * nS * 30},
])
plan["часов"] = (plan["прогонов"] / RUNS_PER_HOUR).round(1)
SCREENING_DESIGN = "full" if plan.loc[0, "часов"] <= SCREEN_BUDGET_H else "half_resV"
print(f"Пропускная способность: {RUNS_PER_HOUR:.0f} прогонов/ч на {N_WORKERS} процессах")
print("Решение для скрининга:", "полный факториал 2^5" if SCREENING_DESIGN == "full" else "полуреплика 2^(5-1), разрешение V")
plan

## 6. Базовый прогон P0 - точка нормировки

Все меры выключены, номинальная неопределённость, три уровня шока, первые `R_SCREEN` seed скринингового пула. Эти же прогоны - угол факториального плана этапа 1, повторно их считать не нужно (совпадают `run_id`).

Нормировка (применяем на этапах 1-4): K0, K1, K4, K5 делим на среднее P0 при том же `shock-depth` (и той же точке неопределённости на этапе 3): 1 = как без вмешательства, 0 = ущерба нет. У P0 K2 = K3 = 0, поэтому их на P0 делить нельзя; их приводим к масштабу дефицита P0: K2 / (L4-subsidy · K0(P0)) и K3-volume / K0(P0), то есть «сколько субсидированного импорта / низкосортного бензина на литр несостоявшегося дефицита».

In [ ]:
scr_seeds = SEEDS["screening"][:R_SCREEN]
base_jobs = [job(s, "P0", shock=sh, stage="screening") for sh in SHOCK_LEVELS for s in scr_seeds]
BASE = ar.run_batch(base_jobs, CFG, n_workers=N_WORKERS, link=link, out_csv=OUT_DIR / "baseline_P0.csv")

def ci(x):
    x = np.asarray(x, float)
    h = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return pd.Series({"mean": x.mean(), "ci95": h, "min": x.min(), "max": x.max()})

cols = ["K0", "K1", "K4", "K5_hours", "K0@90", "price", "dry_share"]
BASE_SUMMARY = BASE.groupby("shock")[cols].apply(lambda g: g.apply(ci)).unstack().round(2)
BASE_SUMMARY.to_csv(OUT_DIR / "baseline_P0_summary.csv", encoding="utf-8-sig")
print("Потеря мощности рынка:", BASE.groupby("shock")["capacity_loss"].first().round(3).to_dict())
BASE_SUMMARY

## 7. Фиксация подготовки

Всё, что нужно следующим этапам и разделу Experimental design отчёта: версии, хэш модели, константы, seed, горизонт, R, процессы, решение о плане, итоги проверок. Следующие этапы читают `prep_summary.json` и проверяют, что хэш модели не изменился.

In [ ]:
import importlib.metadata as im
def _pkg_version(name):
    try:
        return im.version(name)
    except im.PackageNotFoundError:
        return "unknown"

PREP = {
    "created": time.strftime("%Y-%m-%d %H:%M"),
    "netlogo_version": NETLOGO_VERSION, "model_file": MODEL_PATH.name, "model_sha256": MODEL_SHA256,
    "python": sys.version.split()[0], "pynetlogo": _pkg_version("pynetlogo"),
    "horizon": HORIZON, "checkpoints": CHECKPOINTS, "shock_levels": SHOCK_LEVELS, "shock_mid": SHOCK_MID,
    "lever_on": LEVER_ON, "lever_ranges": LEVER_RANGES, "unc_ranges": UNC_RANGES, "c3_range": C3_RANGE,
    "interface_defaults": INTERFACE_DEFAULTS, "nominal_uncertain": NOMINAL_UNC, "constants": CONSTANTS,
    "master_seed": MASTER_SEED, "r_screen": R_SCREEN, "delta": DELTA,
    "n_workers": N_WORKERS, "runs_per_hour": RUNS_PER_HOUR, "screening_design": SCREENING_DESIGN,
    "checks": CHECKS,
}
(OUT_DIR / "prep_summary.json").write_text(json.dumps(PREP, indent=1, ensure_ascii=False, default=float), encoding="utf-8")
n_fail = sum(not c["ok"] for c in CHECKS.values())
print(f"Проверок: {len(CHECKS)}, не пройдено: {n_fail}")
print("Файлы:", sorted(p.name for p in OUT_DIR.iterdir()))